# 13. Checkpoint loss sweep — kin_gamma0, sg_k3_fresh, ddpm_seed42, ddrm_prior

Consolidates the loss-fn sweep (RULES.md #4: MAE/wavelet/starlet/gradient in place of the
original objective, one variable changed at a time) across the four best_models checkpoints
that don't share `05-unet-line-emission.ipynb`'s 1-channel line-emission U-Net shape --
`winner_aug_seed43`, `winner_p10_seed44` and `winner_beam_seed42` stay in 05, which already
has their data plumbing. This notebook covers what needs different data or a different
architecture entirely:

| checkpoint | shape | domain | section |
|---|---|---|---|
| `kin_gamma0` | 31-ch spectral U-Net | line-emission | 2 |
| `sg_k3_fresh` | 7-ch spectral U-Net | self-gravitating | 3 |
| `ddpm_seed42` | conditional DDPM | line-emission | 4 |
| `ddrm_prior` | unconditional DDPM | self-gravitating | 5 |

For the two U-Net sections, MAE/wavelet/starlet/gradient are the exact same losses as 05
(`src/utils/losses.py`, `LOSS_REGISTRY`). For the two diffusion sections, "loss sweep" is a
different question -- there is no direct pixel loss to swap, only a noise/v-prediction
residual -- so it means `loss_type='l1'` (the diffusion analogue of MAE, on the residual
these models actually regress) plus an optional wavelet/starlet/gradient term on the
predicted-clean estimate x0_hat (`src/training/diffusion.py`, `noise_estimation_loss`,
2026-09-15). Designed, not mechanically reused -- see that function's docstring.

Every arm is fine-tuned from its checkpoint (min_epochs=30) AND trained fresh
(min_epochs=50), same ablation split as 05: does the loss help a converged model, does
it help from scratch, and are those the same answer.


## 0. Bootstrap

In [1]:
import os, sys, subprocess, glob, re

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'midterm-prep'
if ON_KAGGLE:
    REPO = '/kaggle/working/EXXA'; PKG = os.path.join(REPO, 'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    'pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks')); sys.path.insert(0, PKG)

    # Line-emission cubes: run_<id>_<step>_rt_<pp> folders (same pattern as 05/06/08).
    run_re = re.compile(r'run_\d+_\d+_rt_\d+', re.I)
    roots = {}
    for p in glob.glob('/kaggle/input/**/run_*', recursive=True):
        if os.path.isdir(p) and run_re.search(os.path.basename(p)):
            roots[os.path.dirname(p)] = roots.get(os.path.dirname(p), 0) + 1
    if not roots:
        raise FileNotFoundError('No run_<id>_<step>_rt_<pp> folders under /kaggle/input. '
                                'Attach the line-emission Dataset.')
    DATA_DIR = max(roots, key=roots.get)

    # Self-gravitating cubes: run_9*_rt_* (notebook 12's pattern -- SG run IDs start 9xxx).
    sg_hits = [p for p in glob.glob('/kaggle/input/**/run_9*_rt_*', recursive=True)
              if os.path.isdir(p)]
    SG_DATA_DIR = os.path.dirname(sg_hits[0]) if sg_hits else None

    # best_models checkpoints: uploaded as a Dataset under a NEUTRAL extension (RULES.md
    # #3 -- a .pth IS a zip, Kaggle unpacks it into a directory on dataset upload, and
    # torch.load then fails with 'Is a directory'). Matched by filename stem, any of
    # .pth/.ckpt/.pth.tar, so the exact upload extension doesn't matter here.
    def locate_ckpt(stem):
        hits = [h for ext in ('.pth', '.ckpt', '.pth.tar')
               for h in glob.glob(f'/kaggle/input/**/{stem}{ext}', recursive=True)
               if os.path.isfile(h)]
        return hits[0] if hits else None
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'):
        os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../Line Emission Data'
    SG_DATA_DIR = '../self-gravitating cube and dirty cube/sg_synth'
    def locate_ckpt(stem):
        hits = glob.glob(f'../models/best_models/{stem}.pth')
        return hits[0] if hits else None

print('DATA_DIR   :', DATA_DIR)
print('SG_DATA_DIR:', SG_DATA_DIR or 'NOT FOUND -- section 3/5 will be skipped')


Cloning into '/kaggle/working/EXXA'...
Updating files: 100% (5489/5489), done.


DATA_DIR   : /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data
SG_DATA_DIR: /kaggle/input/datasets/krishanyadav333/exxa-sg-synth-pairs/kaggle-sg-training-dataset


## 0b. Pull latest `src` (re-run anytime -- no kernel restart needed)

In [2]:
if ON_KAGGLE:
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','FETCH_HEAD'], check=True)
    print(subprocess.run(['git','-C',REPO,'log','--oneline','-1'], capture_output=True, text=True).stdout)
    import importlib, src; importlib.reload(src)


From https://github.com/KrishanYadav333/EXXA
 * branch            midterm-prep -> FETCH_HEAD


HEAD is now at 80a199f fix: RAM leak root cause found -- DataLoader fork-storm, persistent_workers=True
80a199f fix: RAM leak root cause found -- DataLoader fork-storm, persistent_workers=True



## 1. Imports, device, shared config

In [3]:
import time, csv, shutil, math
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data.cube_split import split_cubes, list_cubes
from src.data.fits_cube_dataset import FITSChannelDataset
from src.training.sweep import train_unet, LOSS_REGISTRY

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU = torch.cuda.device_count()
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

LOSSES = ['mae', 'wavelet', 'starlet', 'gradient']   # the four U-Net arms, every section

# Host RAM ran out 3.3h into Version 3 (epoch times climbing 110s -> 406s first) -- a slow
# leak not yet located. Stop training NEW arms after this many per session, counted across
# ALL sections; attach this version's Output as an Input and the next session resumes.
# Version 3 finished 2 arms and died in the 3rd. 0 = no cap.
MAX_NEW_ARMS_PER_SESSION = 2
_new_arms_trained = 0
# Fine-tune arms train at this fraction of the from-scratch lr (notebook 10's convention).
# Version 3's fine-tune arms at full lr spiked at epoch 3 (mae val 0.0096 -> 0.0369, wavelet
# 0.0005 -> 0.0055): the pretrained weights get knocked out and fine-tuning becomes a worse
# from-scratch run.
FINETUNE_LR_SCALE = 0.1


def _arm_tag(source):
    # 'ft', not 'finetune': Version 3's kin_gamma0_mae_finetune trained at full lr before
    # FINETUNE_LR_SCALE existed. A new name keeps that row/checkpoint as its own record
    # (RULES.md #12) instead of resume silently counting it as the current recipe.
    return 'ft' if source == 'finetune' else source


def _cap_reached(name):
    if MAX_NEW_ARMS_PER_SESSION and _new_arms_trained >= MAX_NEW_ARMS_PER_SESSION:
        print(f'--- {name}: DEFERRED, session cap of {MAX_NEW_ARMS_PER_SESSION} new arms '
              f'reached -- resumes next session ---', flush=True)
        return True
    return False

OUT_DIR = '../results'
os.makedirs(OUT_DIR, exist_ok=True)
CKPT_DIR = '../results/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)


def persist_ckpt(path, note='', csv_path=None):
    """Copy a finished checkpoint (and the CSV row that describes it) to /kaggle/working
    the moment it exists (RULES.md #1). CKPT_DIR/OUT_DIR are inside the git clone, wiped by
    the next session's `git reset --hard`, and are NOT part of the notebook Output -- only
    /kaggle/working survives. `csv_path` is explicit rather than looked up by name because
    this is defined once here but called from four different sections, each with its own
    CSV var (KIN_CSV/SG_CSV/DDPM_CSV/DDRM_CSV) that doesn't exist yet when THIS function is
    defined -- passing it avoids a NameError on whichever CSVs haven't been reached yet."""
    if not (ON_KAGGLE and path and os.path.exists(path)):
        return None
    base = os.path.basename(path)
    dst = os.path.join('/kaggle/working', base[:-4] if base.endswith('.pth.tar') else base)
    shutil.copy2(path, dst)
    if csv_path and os.path.exists(csv_path):
        shutil.copy2(csv_path, os.path.join('/kaggle/working', os.path.basename(csv_path)))
    print(f'    [persisted] {os.path.basename(dst)} ({os.path.getsize(dst)/1e6:.0f} MB)'
         + (f' -- {note}' if note else ''), flush=True)
    return dst


def _import_prior_nb13():
    """Restore checkpoints + CSV rows from an earlier session's Output, same reasoning as
    05's `_import_prior_nb05`: without this, attaching a crashed or finished session's
    Output as an Input to the next session changes nothing, because CKPT_DIR/OUT_DIR are
    wiped by THIS session's own bootstrap before this cell ever runs."""
    if not ON_KAGGLE:
        return
    n_ck = 0
    for ext in ('.pth', '.ckpt', '.pth.tar'):
        for src in sorted(glob.glob(f'/kaggle/input/**/nb13_*{ext}', recursive=True)):
            if not os.path.isfile(src):
                continue
            dst = os.path.join(CKPT_DIR, os.path.basename(src)[:-len(ext)] + '.pth')
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                n_ck += 1
    n_csv = 0
    for csv_name in ('nb13_kin_loss_sweep.csv', 'nb13_sg_loss_sweep.csv',
                     'nb13_ddpm_loss_sweep.csv', 'nb13_ddrm_loss_sweep.csv'):
        dst = os.path.join(OUT_DIR, csv_name)
        if os.path.exists(dst):
            continue
        hits = glob.glob(f'/kaggle/input/**/{csv_name}', recursive=True)
        if hits:
            shutil.copy2(hits[0], dst)
            n_csv += 1
    if n_ck or n_csv:
        print(f'[nb13 prior] restored {n_ck} checkpoint(s) and {n_csv} CSV(s) from a prior Output')


_import_prior_nb13()

print(f'device: {device} | GPUs: {N_GPU} | losses: {LOSSES}')


[nb13 prior] restored 26 checkpoint(s) and 4 CSV(s) from a prior Output
device: cuda | GPUs: 2 | losses: ['mae', 'wavelet', 'starlet', 'gradient']


## 2. Kinematic -- `kin_gamma0` loss sweep

31-channel spectral context (k=15, line FWHM ~37 channels), `kinematic_gamma=0` -- the
strongest in-domain wiggle result in the project (models/best_models/README.md: 0.8155 +/-
0.2106 across 5 holdouts). gamma stays 0 in every arm below: swapping loss_name is the only
change, so a wiggle regression can't be blamed on a kinematic term that isn't there.


In [4]:
K_KIN = 15
N_CH_KIN = 2 * K_KIN + 1
TARGET_SIZE_KIN = 256
N_SAMPLES_KIN = 100
NW = 2 if torch.cuda.is_available() else 0

KIN_WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
                  lr=8.196504330730313e-4, alpha=0.8877681051398497,
                  sched_patience=8, batch_size=4,
                  n_neighbors=K_KIN, out_channels=N_CH_KIN, kinematic_gamma=0.0)

train_cubes_kin, val_cubes_kin, _ = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                val_fraction=0.2, seed=SEED)
_kw = dict(n_samples=N_SAMPLES_KIN, target_size=TARGET_SIZE_KIN, seed=SEED,
          subtract_continuum=True, continuum_n=5,
          n_neighbors=K_KIN, stack_target=True, verbose=False)
train_ds_kin = FITSChannelDataset(train_cubes_kin, **_kw)
val_ds_kin   = FITSChannelDataset(val_cubes_kin, **_kw)
d, c = train_ds_kin[0]
assert d.shape == c.shape == (N_CH_KIN, TARGET_SIZE_KIN, TARGET_SIZE_KIN)
print(f'kin: train {len(train_ds_kin)} | val {len(val_ds_kin)} | {N_CH_KIN}-channel stacks')

# Channel velocities, needed by KinematicLoss when gamma>0 -- unused here (gamma=0) but
# train_unet's kinematic_gamma branch still checks velax_kms is not None if gamma>0, and
# kin_gamma0's OWN checkpoint was trained carrying this, so keep the config identical.
from astropy.io import fits
_h = fits.getheader(train_cubes_kin[0]['clean'])
VELAX_KIN = (np.arange(N_CH_KIN) - K_KIN) * float(_h['CDELT3'])


CUBE-LEVEL SPLIT (grouped by RunID — no channel-level leakage)
  data_dir          : /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data
  total cubes       : 14  across 11 distinct RunIDs
  seed=42  n_holdout=3  val_fraction=0.2
----------------------------------------------------------------------
  TRAIN   :  7 cubes | RunIDs ['0006', '0010', '0020', '0022', '0030', '0035']
  VAL     :  2 cubes | RunIDs ['0016', '0036']
  HOLDOUT :  5 cubes | RunIDs ['0002', '0025', '0026']  <-- inference only, NEVER trained/validated
----------------------------------------------------------------------
  HOLDOUT cube folders (reserved for moment-map evaluation):
    - run_0002_00560_rt_00
    - run_0002_00560_rt_01
    - run_0002_00560_rt_04
    - run_0025_01000_rt_04
    - run_0026_00005_rt_04
kin: train 700 | val 200 | 31-channel stacks


In [5]:
KIN_CKPT_SRC = locate_ckpt('kin_gamma0')
print('kin_gamma0 source:', KIN_CKPT_SRC or 'NOT FOUND -- fine-tune arms fall back to fresh init')

KIN_FIELDS = ['config', 'source', 'psnr', 'ssim', 'mse', 'best_val_loss', 'best_epoch',
             'epochs_run', 'wall_time_s']
KIN_CSV = os.path.join(OUT_DIR, 'nb13_kin_loss_sweep.csv')


def _done_rows(path, fields, ext='.pth'):
    """Rows whose CHECKPOINT is also present. A CSV row on its own would mark an arm done
    with no weights behind it: never retrained, and the model gone for good (RULES.md #12).
    That combination is reachable whenever a CSV is restored but its .pth is not -- e.g.
    recovering a failed version's Output by hand, where Kaggle's Add Input cannot attach a
    failed run and the files have to be re-uploaded piecemeal."""
    out = {}
    if os.path.exists(path):
        with open(path, newline='') as f:
            for r in csv.DictReader(f):
                name = r['config']
                if not os.path.exists(os.path.join(CKPT_DIR, f'nb13_{name}{ext}')):
                    print(f'[resume] {name}: CSV row found but no checkpoint -- will retrain')
                    continue
                out[name] = {k: r.get(k, '') for k in fields}
    return out


kin_done = _done_rows(KIN_CSV, KIN_FIELDS)
if kin_done:
    print(f'[resume] {len(kin_done)} kin arm(s) already scored: {sorted(kin_done)}')

kin_rows = list(kin_done.values())
for loss_name in LOSSES:
    for source in ('finetune', 'fresh'):
        name = f'kin_gamma0_{loss_name}_{_arm_tag(source)}'
        if name in kin_done:
            print(f'--- {name}: SKIPPED, already done ---')
            continue
        if _cap_reached(name):
            continue
        ckpt = os.path.join(CKPT_DIR, f'nb13_{name}.pth')
        init_state = None
        if source == 'finetune':
            if KIN_CKPT_SRC is None:
                print(f'--- {name}: SKIPPED, no source checkpoint to fine-tune from ---')
                continue
            init_state = torch.load(KIN_CKPT_SRC, map_location=device,
                                    weights_only=False)['model_state_dict']
            _min_ep = 30
        else:
            _min_ep = 50
        print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
        res = train_unet(train_ds_kin, val_ds_kin, device, **{**KIN_WINNER, 'loss_name': loss_name,
                            'lr': KIN_WINNER['lr'] * (FINETUNE_LR_SCALE if source == 'finetune' else 1.0)},
                         velax_kms=VELAX_KIN, min_epochs=_min_ep, max_epochs=45, patience=6,
                         num_workers=NW, seed=SEED, ckpt_path=ckpt, verbose=True,
                         init_state_dict=init_state)
        row = {'config': name, 'source': source,
               **{k: res[k] for k in KIN_FIELDS if k in res}}
        kin_rows.append(row)
        new = not os.path.exists(KIN_CSV)
        with open(KIN_CSV, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=KIN_FIELDS)
            if new: w.writeheader()
            w.writerow({k: row.get(k, '') for k in KIN_FIELDS})
        persist_ckpt(ckpt, name, csv_path=KIN_CSV)
        _new_arms_trained += 1
        print(f'  {name}: PSNR {res["psnr"]:.4f} | SSIM {res["ssim"]:.4f}')
        res.pop('model', None)
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\nkin_gamma0 sweep: {len(kin_rows)} row(s)')


kin_gamma0 source: /kaggle/input/datasets/krishanyadav333/exxa-13-checkpoint-sources/kin_gamma0.ckpt
[resume] 8 kin arm(s) already scored: ['kin_gamma0_gradient_fresh', 'kin_gamma0_gradient_ft', 'kin_gamma0_mae_fresh', 'kin_gamma0_mae_ft', 'kin_gamma0_starlet_fresh', 'kin_gamma0_starlet_ft', 'kin_gamma0_wavelet_fresh', 'kin_gamma0_wavelet_ft']
--- kin_gamma0_mae_ft: SKIPPED, already done ---
--- kin_gamma0_mae_fresh: SKIPPED, already done ---
--- kin_gamma0_wavelet_ft: SKIPPED, already done ---
--- kin_gamma0_wavelet_fresh: SKIPPED, already done ---
--- kin_gamma0_starlet_ft: SKIPPED, already done ---
--- kin_gamma0_starlet_fresh: SKIPPED, already done ---
--- kin_gamma0_gradient_ft: SKIPPED, already done ---
--- kin_gamma0_gradient_fresh: SKIPPED, already done ---

kin_gamma0 sweep: 8 row(s)


## 3. Self-gravitating -- `sg_k3_fresh` loss sweep

7-channel spectral context (k=3), the only self-gravitating-trained checkpoint in the
project to beat doing nothing on the wiggle across a genuine holdout (0.681 vs dirty's own
0.594). Different domain from every other checkpoint here -- scientifically a stretch to
expect the same loss fix to transfer, which is exactly why it's worth checking rather than
assuming (RULES.md #6: a spread across cubes here is not the same claim as one across
seeds elsewhere).


In [6]:
if SG_DATA_DIR is None:
    print('SG_DATA_DIR not found -- section 3 skipped. Attach the self-gravitating Dataset.')
else:
    K_SG = 3
    TARGET_SIZE_SG = 256
    N_SAMPLES_SG = 120

    SG_WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
                     lr=8.196504330730313e-4, alpha=0.8877681051398497,
                     sched_patience=8, batch_size=8,
                     n_neighbors=K_SG, out_channels=1)   # stack_target=False -> 1-ch target

    # Notebook 12's own split: 3 train RunIDs, 1 val, held out from the wiggle-scoring set.
    TRAIN_RUNS_SG = ['run_9015_00370_rt_00', 'run_9019_00019_rt_00', 'run_9032_00020_rt_00']
    VAL_RUN_SG = 'run_9025_00370_rt_00'

    all_cubes_sg = {c['folder']: c for c in list_cubes(SG_DATA_DIR)}
    train_cubes_sg = [all_cubes_sg[r] for r in TRAIN_RUNS_SG]
    val_cubes_sg = [all_cubes_sg[VAL_RUN_SG]]

    _kw = dict(n_samples=N_SAMPLES_SG, target_size=TARGET_SIZE_SG, seed=SEED,
              subtract_continuum=False, n_neighbors=K_SG, stack_target=False, verbose=False)
    train_ds_sg = FITSChannelDataset(train_cubes_sg, **_kw)
    val_ds_sg   = FITSChannelDataset(val_cubes_sg, **_kw)
    d, c = train_ds_sg[0]
    assert d.shape[0] == 2 * K_SG + 1 and c.shape[0] == 1
    print(f'sg: train {len(train_ds_sg)} | val {len(val_ds_sg)} | {2*K_SG+1}-channel input, 1-ch target')


sg: train 360 | val 120 | 7-channel input, 1-ch target


In [7]:
if SG_DATA_DIR is None:
    print('section 3 skipped')
else:
    SG_CKPT_SRC = locate_ckpt('sg_k3_fresh')
    print('sg_k3_fresh source:', SG_CKPT_SRC or 'NOT FOUND -- fine-tune arms fall back to fresh init')

    SG_FIELDS = ['config', 'source', 'psnr', 'ssim', 'mse', 'best_val_loss', 'best_epoch',
                'epochs_run', 'wall_time_s']
    SG_CSV = os.path.join(OUT_DIR, 'nb13_sg_loss_sweep.csv')
    sg_done = _done_rows(SG_CSV, SG_FIELDS)
    if sg_done:
        print(f'[resume] {len(sg_done)} sg arm(s) already scored: {sorted(sg_done)}')

    sg_rows = list(sg_done.values())
    for loss_name in LOSSES:
        for source in ('finetune', 'fresh'):
            name = f'sg_k3_{loss_name}_{_arm_tag(source)}'
            if name in sg_done:
                print(f'--- {name}: SKIPPED, already done ---')
                continue
            if _cap_reached(name):
                continue
            ckpt = os.path.join(CKPT_DIR, f'nb13_{name}.pth')
            init_state = None
            if source == 'finetune':
                if SG_CKPT_SRC is None:
                    print(f'--- {name}: SKIPPED, no source checkpoint to fine-tune from ---')
                    continue
                init_state = torch.load(SG_CKPT_SRC, map_location=device,
                                        weights_only=False)['model_state_dict']
                _min_ep = 30
            else:
                _min_ep = 50
            print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
            res = train_unet(train_ds_sg, val_ds_sg, device, **{**SG_WINNER, 'loss_name': loss_name,
                                'lr': SG_WINNER['lr'] * (FINETUNE_LR_SCALE if source == 'finetune' else 1.0)},
                             min_epochs=_min_ep, max_epochs=60, patience=8,
                             num_workers=NW, seed=SEED, ckpt_path=ckpt, verbose=True,
                             init_state_dict=init_state)
            row = {'config': name, 'source': source,
                   **{k: res[k] for k in SG_FIELDS if k in res}}
            sg_rows.append(row)
            new = not os.path.exists(SG_CSV)
            with open(SG_CSV, 'a', newline='') as f:
                w = csv.DictWriter(f, fieldnames=SG_FIELDS)
                if new: w.writeheader()
                w.writerow({k: row.get(k, '') for k in SG_FIELDS})
            persist_ckpt(ckpt, name, csv_path=SG_CSV)
            _new_arms_trained += 1
            print(f'  {name}: PSNR {res["psnr"]:.4f} | SSIM {res["ssim"]:.4f}')
            res.pop('model', None)
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'\nsg_k3_fresh sweep: {len(sg_rows)} row(s)')


sg_k3_fresh source: /kaggle/input/datasets/krishanyadav333/exxa-13-checkpoint-sources/sg_k3_fresh.ckpt
[resume] 8 sg arm(s) already scored: ['sg_k3_gradient_fresh', 'sg_k3_gradient_ft', 'sg_k3_mae_fresh', 'sg_k3_mae_ft', 'sg_k3_starlet_fresh', 'sg_k3_starlet_ft', 'sg_k3_wavelet_fresh', 'sg_k3_wavelet_ft']
--- sg_k3_mae_ft: SKIPPED, already done ---
--- sg_k3_mae_fresh: SKIPPED, already done ---
--- sg_k3_wavelet_ft: SKIPPED, already done ---
--- sg_k3_wavelet_fresh: SKIPPED, already done ---
--- sg_k3_starlet_ft: SKIPPED, already done ---
--- sg_k3_starlet_fresh: SKIPPED, already done ---
--- sg_k3_gradient_ft: SKIPPED, already done ---
--- sg_k3_gradient_fresh: SKIPPED, already done ---

sg_k3_fresh sweep: 8 row(s)


## 4. Diffusion -- `ddpm_seed42` loss sweep

Conditional DDPM, v-prediction + cosine schedule (notebook 06's winning objective, 19.9 dB
of the sweep spread over eps/linear). There is no direct pixel loss here to swap for
MAE/wavelet/starlet/gradient -- the model regresses a noise/v RESIDUAL, not an image. Two
kinds of arm instead (`src/training/diffusion.py`, `noise_estimation_loss`, 2026-09-15):

- `loss_type='l1'` -- the direct diffusion analogue of MAE: absolute instead of squared
  error on the residual itself.
- `aux_loss_name` -- keeps `loss_type='l2'` (the original objective) and ADDS a
  wavelet/starlet/gradient detail term on the predicted-clean estimate x0_hat, inverted from
  the noise/v prediction by the standard DDPM identity. This is closer in spirit to what the
  U-Net sweep does, since it acts on an actual image, not a residual.

**`AUX_WEIGHT` below is a rough order-of-magnitude guess, not a tuned value** -- the primary
loss SUMS squared error over every pixel (`per_sample.sum(dim=(1,2,3))`, ~65,536 terms at
256px), while the aux functions are per-pixel MEANS, so a weight of 1 would make the aux
term invisible. Chosen so its gradient is the same rough order as the primary term; inspect
the printed train/val loss and adjust if the aux term is clearly negligible or dominant.


In [8]:
from src.data.stacked_pair import StackedPairDataset
from src.models.diffusion_unet import default_diffusion_config
from src.training.diffusion import DenoisingDiffusion

TARGET_SIZE_DDPM = 256
BATCH_SIZE_DDPM = 8
LR_DDPM = 2e-4   # notebook 06's from-scratch lr; fine-tune arms use FINETUNE_LR_SCALE of it
AUX_WEIGHT = 2000.0   # see markdown above -- an order-of-magnitude guess, not tuned

train_cubes_ddpm, val_cubes_ddpm, _ = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                   val_fraction=0.2, seed=SEED)
_kw = dict(n_samples=100, target_size=TARGET_SIZE_DDPM, seed=SEED,
          subtract_continuum=True, continuum_n=5, verbose=False)
_train_pairs = FITSChannelDataset(train_cubes_ddpm, **_kw)
_val_pairs   = FITSChannelDataset(val_cubes_ddpm, **_kw)
ddpm_train_ds = StackedPairDataset(_train_pairs)
ddpm_val_ds   = StackedPairDataset(_val_pairs)
_nw = 4 if ON_KAGGLE else 0
ddpm_train_loader = DataLoader(ddpm_train_ds, batch_size=BATCH_SIZE_DDPM, shuffle=True,
                               num_workers=_nw, pin_memory=True)
ddpm_val_loader = DataLoader(ddpm_val_ds, batch_size=BATCH_SIZE_DDPM, shuffle=False,
                             num_workers=_nw, pin_memory=True)
print(f'ddpm: train {len(ddpm_train_ds)} | val {len(ddpm_val_ds)} items')


def make_ddpm_cfg(loss_type='l2', aux_loss_name=None, aux_weight=0.0):
    c = default_diffusion_config(image_size=TARGET_SIZE_DDPM)
    c.model.ch_mult = [1, 2, 2, 2, 4]
    c.model.ema_rate = 0.99
    c.diffusion.prediction_type = 'v'
    c.diffusion.beta_schedule = 'cosine'
    c.diffusion.min_snr_gamma = 0.0
    c.diffusion.loss_type = loss_type
    c.diffusion.aux_loss_name = aux_loss_name
    c.diffusion.aux_weight = aux_weight
    return c


CUBE-LEVEL SPLIT (grouped by RunID — no channel-level leakage)
  data_dir          : /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data
  total cubes       : 14  across 11 distinct RunIDs
  seed=42  n_holdout=3  val_fraction=0.2
----------------------------------------------------------------------
  TRAIN   :  7 cubes | RunIDs ['0006', '0010', '0020', '0022', '0030', '0035']
  VAL     :  2 cubes | RunIDs ['0016', '0036']
  HOLDOUT :  5 cubes | RunIDs ['0002', '0025', '0026']  <-- inference only, NEVER trained/validated
----------------------------------------------------------------------
  HOLDOUT cube folders (reserved for moment-map evaluation):
    - run_0002_00560_rt_00
    - run_0002_00560_rt_01
    - run_0002_00560_rt_04
    - run_0025_01000_rt_04
    - run_0026_00005_rt_04
ddpm: train 700 | val 200 items


In [9]:
DDPM_CKPT_SRC = locate_ckpt('ddpm_seed42')
print('ddpm_seed42 source:', DDPM_CKPT_SRC or 'NOT FOUND -- fine-tune arms fall back to fresh init')

DDPM_ARMS = ([('l1', None, 0.0)] +
            [('l2', name, AUX_WEIGHT) for name in ('wavelet', 'starlet', 'gradient')])

DDPM_FIELDS = ['config', 'source', 'loss_type', 'aux_loss_name', 'psnr', 'ssim', 'mse',
              'best_val_loss', 'wall_time_s']
DDPM_CSV = os.path.join(OUT_DIR, 'nb13_ddpm_loss_sweep.csv')
ddpm_done = _done_rows(DDPM_CSV, DDPM_FIELDS)
if ddpm_done:
    print(f'[resume] {len(ddpm_done)} ddpm arm(s) already scored: {sorted(ddpm_done)}')

ddpm_rows = list(ddpm_done.values())
for loss_type, aux_name, aux_w in DDPM_ARMS:
    tag = aux_name or loss_type
    for source in ('finetune', 'fresh'):
        name = f'ddpm_{tag}_{_arm_tag(source)}'
        if name in ddpm_done:
            print(f'--- {name}: SKIPPED, already done ---')
            continue
        if source == 'finetune' and DDPM_CKPT_SRC is None:
            print(f'--- {name}: SKIPPED, no source checkpoint to fine-tune from ---')
            continue
        if _cap_reached(name):
            continue
        ckpt = os.path.join(CKPT_DIR, f'nb13_{name}.pth.tar')
        print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
        t0 = time.time()
        d = DenoisingDiffusion(config=make_ddpm_cfg(loss_type, aux_name, aux_w),
                               device=str(device),
                               lr=LR_DDPM * (FINETUNE_LR_SCALE if source == 'finetune' else 1.0),
                               checkpoint_path=ckpt,
                               data_parallel=False)
        n_epochs = 30
        if source == 'finetune':
            d.load_checkpoint(DDPM_CKPT_SRC)
            # RULES.md #4: best_val_loss carried over is on the OLD objective's scale (l1
            # vs l2 vs l2+aux are not comparable numbers), so a stale bound here would mean
            # save_checkpoint(best=True) never fires under the new objective and nothing
            # gets persisted. Reset so this run tracks its own best from scratch.
            d.best_val_loss = float('inf')
        else:
            n_epochs = 50   # fresh init needs more than 30 epochs to be a meaningful arm
        d.train(ddpm_train_loader, ddpm_val_loader, n_epochs=n_epochs, verbose=True)
        m = d.evaluate(ddpm_val_loader, sampling_timesteps=25, use_ema=True, n_avg=4)
        row = {'config': name, 'source': source, 'loss_type': loss_type,
              'aux_loss_name': aux_name or '', 'psnr': round(float(m['psnr']), 4),
              'ssim': round(float(m['ssim']), 5), 'mse': round(float(m['mse']), 8),
              'best_val_loss': round(float(d.best_val_loss), 5),
              'wall_time_s': round(time.time() - t0, 1)}
        ddpm_rows.append(row)
        new = not os.path.exists(DDPM_CSV)
        with open(DDPM_CSV, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=DDPM_FIELDS)
            if new: w.writeheader()
            w.writerow({k: row.get(k, '') for k in DDPM_FIELDS})
        persist_ckpt(ckpt, name, csv_path=DDPM_CSV)
        _new_arms_trained += 1
        print(f'  {name}: PSNR {row["psnr"]:.4f} | SSIM {row["ssim"]:.4f}')
        del d
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\nddpm_seed42 sweep: {len(ddpm_rows)} row(s)')


ddpm_seed42 source: /kaggle/input/datasets/krishanyadav333/exxa-13-checkpoint-sources/ddpm_seed42.ckpt
[resume] 8 ddpm arm(s) already scored: ['ddpm_gradient_fresh', 'ddpm_gradient_ft', 'ddpm_l1_fresh', 'ddpm_l1_ft', 'ddpm_starlet_fresh', 'ddpm_starlet_ft', 'ddpm_wavelet_fresh', 'ddpm_wavelet_ft']
--- ddpm_l1_ft: SKIPPED, already done ---
--- ddpm_l1_fresh: SKIPPED, already done ---
--- ddpm_wavelet_ft: SKIPPED, already done ---
--- ddpm_wavelet_fresh: SKIPPED, already done ---
--- ddpm_starlet_ft: SKIPPED, already done ---
--- ddpm_starlet_fresh: SKIPPED, already done ---
--- ddpm_gradient_ft: SKIPPED, already done ---
--- ddpm_gradient_fresh: SKIPPED, already done ---

ddpm_seed42 sweep: 8 row(s)


## 5. Diffusion -- `ddrm_prior` loss sweep

Unconditional prior p(clean) -- the model DDRM's reverse sampler restores through, trained
on clean channel maps only (`CleanOnly`, no dirty conditioning). Same `loss_type`/
`aux_loss_name` mechanism as section 4, just `conditional=False`. This is currently
`models/best_models/README.md`'s explicit negative result -- worst of 4 methods in every
wiggle confirmation to date -- so a meaningful outcome here is either arm helping enough to
reopen the question, not necessarily beating the U-Net arms.


In [10]:
if SG_DATA_DIR is None:
    print('SG_DATA_DIR not found -- section 5 skipped.')
else:
    TARGET_SIZE_DDRM = 256
    BATCH_SIZE_DDRM = 8
    LR_DDRM = 2e-5   # notebook 07's from-scratch prior lr; fine-tune arms use FINETUNE_LR_SCALE of it

    train_cubes_ddrm, val_cubes_ddrm, _ = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                       val_fraction=0.2, seed=SEED)
    _kw = dict(n_samples=150, target_size=TARGET_SIZE_DDRM, seed=SEED,
              subtract_continuum=True, continuum_n=5, verbose=False)
    _train_pairs_ddrm = FITSChannelDataset(train_cubes_ddrm, **_kw)
    _val_pairs_ddrm   = FITSChannelDataset(val_cubes_ddrm, **_kw)

    class CleanOnly(torch.utils.data.Dataset):
        """(dirty, clean) -> clean. The prior never sees a dirty image."""
        def __init__(self, pairs): self.pairs = pairs
        def __len__(self): return len(self.pairs)
        def __getitem__(self, i): return self.pairs[i][1]

    ddrm_train_ds, ddrm_val_ds = CleanOnly(_train_pairs_ddrm), CleanOnly(_val_pairs_ddrm)
    _nw = 4 if ON_KAGGLE else 0
    ddrm_train_loader = DataLoader(ddrm_train_ds, batch_size=BATCH_SIZE_DDRM, shuffle=True,
                                   num_workers=_nw, pin_memory=True, drop_last=True)
    ddrm_val_loader = DataLoader(ddrm_val_ds, batch_size=BATCH_SIZE_DDRM, shuffle=False,
                                 num_workers=_nw)
    print(f'ddrm prior: train {len(ddrm_train_ds)} | val {len(ddrm_val_ds)} images')

    def make_ddrm_cfg(loss_type='l2', aux_loss_name=None, aux_weight=0.0):
        c = default_diffusion_config(image_size=TARGET_SIZE_DDRM)
        c.data.conditional = False
        c.diffusion.prediction_type = 'v'
        c.diffusion.beta_schedule = 'cosine'
        c.diffusion.min_snr_gamma = 5.0
        c.diffusion.loss_type = loss_type
        c.diffusion.aux_loss_name = aux_loss_name
        c.diffusion.aux_weight = aux_weight
        return c


CUBE-LEVEL SPLIT (grouped by RunID — no channel-level leakage)
  data_dir          : /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data
  total cubes       : 14  across 11 distinct RunIDs
  seed=42  n_holdout=3  val_fraction=0.2
----------------------------------------------------------------------
  TRAIN   :  7 cubes | RunIDs ['0006', '0010', '0020', '0022', '0030', '0035']
  VAL     :  2 cubes | RunIDs ['0016', '0036']
  HOLDOUT :  5 cubes | RunIDs ['0002', '0025', '0026']  <-- inference only, NEVER trained/validated
----------------------------------------------------------------------
  HOLDOUT cube folders (reserved for moment-map evaluation):
    - run_0002_00560_rt_00
    - run_0002_00560_rt_01
    - run_0002_00560_rt_04
    - run_0025_01000_rt_04
    - run_0026_00005_rt_04
ddrm prior: train 1050 | val 300 images


In [11]:
if SG_DATA_DIR is None:
    print('section 5 skipped')
else:
    DDRM_CKPT_SRC = locate_ckpt('ddrm_prior')
    print('ddrm_prior source:', DDRM_CKPT_SRC or 'NOT FOUND -- fine-tune arms fall back to fresh init')

    # No aux term here: the prior is UNCONDITIONAL (no dirty/clean pair in one batch), so
    # there is no 'clean' to compare x0_hat against inside noise_estimation_loss's aux
    # branch (it requires `conditional`, x0.shape[1] > 1). l1 vs l2 on the residual is the
    # only axis that applies.
    DDRM_ARMS = [('l1', None, 0.0), ('l2', None, 0.0)]

    DDRM_FIELDS = ['config', 'source', 'loss_type', 'best_val_loss', 'wall_time_s']
    DDRM_CSV = os.path.join(OUT_DIR, 'nb13_ddrm_loss_sweep.csv')
    ddrm_done = _done_rows(DDRM_CSV, DDRM_FIELDS)
    if ddrm_done:
        print(f'[resume] {len(ddrm_done)} ddrm arm(s) already scored: {sorted(ddrm_done)}')

    ddrm_rows = list(ddrm_done.values())
    for loss_type, aux_name, aux_w in DDRM_ARMS:
        for source in ('finetune', 'fresh'):
            name = f'ddrm_{loss_type}_{_arm_tag(source)}'
            if name in ddrm_done:
                print(f'--- {name}: SKIPPED, already done ---')
                continue
            if source == 'finetune' and DDRM_CKPT_SRC is None:
                print(f'--- {name}: SKIPPED, no source checkpoint to fine-tune from ---')
                continue
            if _cap_reached(name):
                continue
            ckpt = os.path.join(CKPT_DIR, f'nb13_{name}.pth.tar')
            print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
            t0 = time.time()
            d = DenoisingDiffusion(config=make_ddrm_cfg(loss_type, aux_name, aux_w),
                                   device=str(device),
                                   lr=LR_DDRM * (FINETUNE_LR_SCALE if source == 'finetune' else 1.0),
                                   checkpoint_path=ckpt,
                                   data_parallel=False)
            n_epochs = 30
            if source == 'finetune':
                d.load_checkpoint(DDRM_CKPT_SRC)
                d.best_val_loss = float('inf')   # RULES.md #4, same reasoning as section 4
            else:
                n_epochs = 50
            d.train(ddrm_train_loader, ddrm_val_loader, n_epochs=n_epochs, verbose=True)
            row = {'config': name, 'source': source, 'loss_type': loss_type,
                  'best_val_loss': round(float(d.best_val_loss), 5),
                  'wall_time_s': round(time.time() - t0, 1)}
            ddrm_rows.append(row)
            new = not os.path.exists(DDRM_CSV)
            with open(DDRM_CSV, 'a', newline='') as f:
                w = csv.DictWriter(f, fieldnames=DDRM_FIELDS)
                if new: w.writeheader()
                w.writerow({k: row.get(k, '') for k in DDRM_FIELDS})
            persist_ckpt(ckpt, name, csv_path=DDRM_CSV)
            _new_arms_trained += 1
            print(f'  {name}: best_val_loss {row["best_val_loss"]:.5f}')
            del d
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'\nddrm_prior sweep: {len(ddrm_rows)} row(s)')
    print('Scoring against the DDRM restoration/wiggle pipeline is a separate step '
         '(notebook 07 section 5-6), not run here -- this cell only trains the priors.')


ddrm_prior source: /kaggle/input/datasets/krishanyadav333/exxa-13-checkpoint-sources/ddrm_prior.ckpt
[resume] 2 ddrm arm(s) already scored: ['ddrm_l1_fresh', 'ddrm_l1_ft']
--- ddrm_l1_ft: SKIPPED, already done ---
--- ddrm_l1_fresh: SKIPPED, already done ---

=== ddrm_l2_ft
=> loaded '/kaggle/input/datasets/krishanyadav333/exxa-13-checkpoint-sources/ddrm_prior.ckpt' (epoch 59, step 7729)
Unconditional DDPM training
  params : 16,884,097
  device : cuda (single device), timesteps: 1000
[epoch  60/89] train 16.8756 | val 29.4871  *best  (235.8s)
[epoch  61/89] train 17.4121 | val 27.1603  *best  (255.0s)
[epoch  62/89] train 17.2752 | val 25.0987  *best  (253.6s)
[epoch  63/89] train 17.5518 | val 26.9680  (255.5s)
[epoch  64/89] train 17.5319 | val 23.7449  *best  (256.2s)
[epoch  65/89] train 16.0875 | val 24.0597  (253.4s)
[epoch  66/89] train 15.1944 | val 24.6276  (253.2s)
[epoch  67/89] train 16.5389 | val 24.1024  (254.3s)
[epoch  68/89] train 15.3550 | val 25.8901  (254.5s)
[epoc

## 6. Collect outputs

In [12]:
from src.evaluation.collect_outputs import collect_outputs

_run_dir = collect_outputs(
    '13-checkpoint-loss-sweep',
    [
        'nb13_kin_loss_sweep.csv',
        'nb13_sg_loss_sweep.csv',
        'nb13_ddpm_loss_sweep.csv',
        'nb13_ddrm_loss_sweep.csv',
    ],
    extra={'losses': LOSSES, 'kin_ckpt_found': KIN_CKPT_SRC is not None,
          'sg_ckpt_found': SG_DATA_DIR is not None and 'SG_CKPT_SRC' in dir() and SG_CKPT_SRC is not None,
          'ddpm_ckpt_found': DDPM_CKPT_SRC is not None,
          'ddrm_ckpt_found': SG_DATA_DIR is not None and 'DDRM_CKPT_SRC' in dir() and DDRM_CKPT_SRC is not None},
)
print('\nAnything listed as NOT FOUND above did not get written this run.')


collected 4 file(s), 0.0 MiB -> /kaggle/working/outputs/13-checkpoint-loss-sweep/2026-09-23T215106_80a199f
       0.00 MiB  nb13_ddpm_loss_sweep.csv
       0.00 MiB  nb13_ddrm_loss_sweep.csv
       0.00 MiB  nb13_kin_loss_sweep.csv
       0.00 MiB  nb13_sg_loss_sweep.csv

Anything listed as NOT FOUND above did not get written this run.
